In [ ]:
# ===================== Study Area Map (Best-Coverage ECOSTRESS Scene) =====================
# - Picks the time slice with the most valid pixels over the Bode basin
# - Cleans NoData (0, -9999, -32768) and out-of-range values
# - Publication layout: GridSpec (separate colorbar), locator inset, north arrow, scale bar
# - Zero overlap between legend, inset, and colorbar
# =========================================================================================

# ---- Imports ----
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray  # activates .rio accessor
import geopandas as gpd
from shapely.geometry import box
from pyproj import Geod
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ---- Paths (update if needed) ----
NC_PATH   = r"Y:\Home\goyal\post_doc_work\ecostress\data\ECO_L2T_LSTE.002_70m_aid0001_32N.nc"
RIV_PATH  = r"F:\goyal_shekhar\ecostress\data\shapefiles\HydroRIVERS_v10_eu_shp\HydroRIVERS_v10_eu_shp\HydroRIVERS_v10_eu.shp"
BASIN_SH  = r"F:\goyal_shekhar\ecostress\data\shapefiles\bode_shp\bode.shp"
OUT_DIR   = r"F:\goyal_shekhar\ecostress\figure\bode"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- Styling helpers ----
warnings.filterwarnings("ignore", message="Converting a CFTimeIndex*")  # quiet cftime->pandas warnings

def nice_rc():
    plt.rcParams.update({
        "figure.dpi": 120,
        "savefig.dpi": 600,
        "axes.linewidth": 0.9,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "font.family": "DejaVu Sans",
        "grid.color": "0.88",
        "grid.linewidth": 0.4
    })

def add_north_arrow(ax, xy=(0.05, 0.86), size=0.07, text="N"):
    x, y = xy
    ax.annotate(text, xy=(x, y + size*0.78), xycoords='axes fraction',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.annotate("", xy=(x, y + size), xycoords='axes fraction',
                xytext=(x, y), textcoords='axes fraction',
                arrowprops=dict(arrowstyle="-|>", lw=1.0))

def add_geodesic_scalebar(ax, length_km=None, loc=(0.04, 0.04)):
    """Draw a geodesic scale bar for geographic CRS."""
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    mid_lat = 0.5*(y0+y1)
    geod = Geod(ellps="WGS84")
    # Aim for ~25% of axis width
    target_lon_span = 0.25*(x1-x0)
    _, _, dist_m = geod.inv(x0, mid_lat, x0 + target_lon_span, mid_lat)
    target_km = dist_m/1000.0
    if length_km is None:
        candidates = np.array([1,2,5,10,20,25,50,100,200])
        length_km = candidates[candidates <= target_km].max() if (candidates<=target_km).any() else candidates.min()

    lon_left = x0 + 0.02*(x1-x0)
    lon_delta = target_lon_span*(length_km/target_km)
    for _ in range(4):
        _, _, d_m = geod.inv(lon_left, mid_lat, lon_left + lon_delta, mid_lat)
        lon_delta *= (length_km*1000.0 / max(d_m, 1e-9))

    fx, fy = loc
    dx = x0 + fx*(x1-x0); dy = y0 + fy*(y1-y0)
    h = 0.01*(y1-y0)
    ax.add_patch(Rectangle((dx, dy), lon_delta, h, facecolor="k", edgecolor="k", lw=0.5))
    ax.text(dx + lon_delta/2, dy + 1.5*h, f"{int(length_km)} km",
            ha="center", va="bottom", fontsize=9, color="k")

def format_deg_ticks(ax, use_degrees=True):
    if not use_degrees:
        ax.grid(True, zorder=1)
        return
    from matplotlib.ticker import FuncFormatter
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{abs(x):.2f}°{'E' if x>=0 else 'W'}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{abs(y):.2f}°{'N' if y>=0 else 'S'}"))
    ax.grid(True, zorder=1)

# ---- Load data ----
nice_rc()
ds = xr.open_dataset(NC_PATH)
lst_all = ds["LST"]

# Handle CRS on the raster
if not hasattr(lst_all, "rio") or lst_all.rio.crs is None:
    crs_wkt = None
    gm = lst_all.attrs.get("grid_mapping", None)
    if gm and gm in ds.data_vars:
        crs_wkt = ds[gm].attrs.get("spatial_ref", None)
    if crs_wkt:
        lst_all = lst_all.rio.write_crs(crs_wkt)
    else:
        warnings.warn("No CRS found on LST; assuming EPSG:4326.")
        lst_all = lst_all.rio.write_crs(4326)

# Read vectors and align to raster CRS
gdf_basin = gpd.read_file(BASIN_SH).to_crs(lst_all.rio.crs)
gdf_riv   = gpd.read_file(RIV_PATH).to_crs(lst_all.rio.crs)

# Robust time conversion (cftime -> pandas)
try:
    time_vals = ds.indexes["time"].to_datetimeindex()
except Exception:
    # Fallback for exotic calendars
    t = ds["time"].values
    time_vals = pd.DatetimeIndex([
        pd.Timestamp(getattr(x, "year", np.nan),
                     getattr(x, "month", 1),
                     getattr(x, "day", 1),
                     getattr(x, "hour", 0),
                     getattr(x, "minute", 0),
                     int(getattr(x, "second", 0))) if x is not None else pd.NaT
        for x in t
    ])

# ---- Choose best-coverage time slice over the basin ----
NODATA_CANDIDATES = [-32768, -9999, 0]
best_i, best_cover = 0, -1

for i in range(lst_all.sizes["time"]):
    arr = lst_all.isel(time=i)
    arr_clip = arr.rio.clip(gdf_basin.geometry, gdf_basin.crs, drop=True)
    a = arr_clip.values.astype(float)
    for nd in NODATA_CANDIDATES:
        a[a == nd] = np.nan
    a[(a < 150) | (a > 400)] = np.nan  # physical screen (K)
    cover = np.isfinite(a).sum()
    if cover > best_cover:
        best_cover = cover
        best_i = i

lst = lst_all.isel(time=best_i)
timestamp = pd.Timestamp(time_vals[best_i]) if len(time_vals) > best_i else None

# Final clip & clean for plotting
lst_clip = lst.rio.clip(gdf_basin.geometry, gdf_basin.crs, drop=True)
a = lst_clip.values.astype(float)
for nd in NODATA_CANDIDATES:
    a[a == nd] = np.nan
a[(a < 150) | (a > 400)] = np.nan
lst_clip_clean = xr.DataArray(
    a, coords=lst_clip.coords, dims=lst_clip.dims, attrs=lst_clip.attrs
)

# Rivers (clipped to padded bbox for speed, then to basin)
minx, miny, maxx, maxy = gdf_basin.total_bounds
padx, pady = 0.02*(maxx-minx), 0.02*(maxy-miny)
bbox = gpd.GeoSeries([box(minx-padx, miny-pady, maxx+padx, maxy+pady)], crs=gdf_basin.crs)
gdf_riv_clip = gpd.clip(gdf_riv, bbox)
gdf_riv_clip = gpd.clip(gdf_riv_clip, gdf_basin)

# ---- Figure layout (no overlap via GridSpec) ----
fig = plt.figure(figsize=(7.4, 5.4))
gs = GridSpec(nrows=1, ncols=2, width_ratios=[1, 0.045], wspace=0.08, figure=fig)
ax  = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])  # dedicated colorbar axis

# Transparent NaNs in colormap
cmap = plt.get_cmap("inferno").copy()
cmap.set_bad(alpha=0.0)

# Robust range from valid pixels only
valid = np.isfinite(a)
vmin, vmax = (np.nanpercentile(a[valid], [2, 98]) if valid.any() else (270, 310))

# Plot raster using xarray's plot (handles coords/orientation); send colorbar to cax
im = lst_clip_clean.plot(
    ax=ax, cmap=cmap, vmin=vmin, vmax=vmax,
    add_colorbar=True, cbar_ax=cax
)
cax.set_ylabel("ECOSTRESS LST [K]")

# Basin & rivers on top
gdf_basin.boundary.plot(ax=ax, color="cyan", linewidth=1.0, zorder=5)
gdf_riv_clip.plot(ax=ax, color="deepskyblue", linewidth=0.7, alpha=0.9, zorder=6)

# Axes/limits/labels
ax.set_xlim(minx-padx, maxx+padx); ax.set_ylim(miny-pady, maxy+pady)
is_geographic = (gdf_basin.crs and gdf_basin.crs.is_geographic)
format_deg_ticks(ax, use_degrees=is_geographic)
ax.set_xlabel("Longitude" if is_geographic else "X")
ax.set_ylabel("Latitude"  if is_geographic else "Y")
title_time = f" (best coverage: {timestamp:%Y-%m-%d %H:%M} UTC)" if timestamp is not None else ""
ax.set_title(f"Bode Basin — ECOSTRESS LST{title_time}", pad=8)

# North arrow & geodesic scale bar (only meaningful for geographic CRS)
add_north_arrow(ax, xy=(0.05, 0.86), size=0.07)
if is_geographic:
    add_geodesic_scalebar(ax, length_km=None, loc=(0.04, 0.04))

# Legend away from inset/scalebar
legend_elems = [
    Line2D([0],[0], color="deepskyblue", lw=1.2, label="HydroRIVERS"),
    Line2D([0],[0], color="cyan", lw=1.2, label="Bode basin"),
]
ax.legend(handles=legend_elems, loc="lower right", frameon=True, framealpha=0.9, borderpad=0.5)

# Locator inset (top-left of main map)
axins = inset_axes(ax, width="26%", height="26%", loc="upper left", borderpad=0.8)
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres")).to_crs(gdf_basin.crs)
eur = world[world["continent"].isin(["Europe"])]
eur.boundary.plot(ax=axins, color="0.55", linewidth=0.4)
world[world["name"].eq("Germany")].plot(ax=axins, facecolor="0.95", edgecolor="0.5", linewidth=0.6)
gpd.GeoSeries([box(minx, miny, maxx, maxy)], crs=gdf_basin.crs)\
   .plot(ax=axins, facecolor="none", edgecolor="crimson", linewidth=1.1)
axins.set_xticks([]); axins.set_yticks([])
axins.set_title("Locator", fontsize=9)

# Panel label (optional)
ax.text(0.01, 0.99, "a", transform=ax.transAxes, ha="left", va="top",
        fontsize=11, fontweight="bold")

plt.tight_layout()

# ---- Save ----
fname = f"study_area_bode_bestcov_{timestamp:%Y%m%d_%H%M}.png" if timestamp is not None else "study_area_bode_bestcov.png"
out_png = os.path.join(OUT_DIR, fname)
#plt.savefig(out_png, bbox_inches="tight")
print(f"✅ Saved: {out_png}   (time index {best_i}, valid pixels: {best_cover})")
plt.show()
